In [1]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px

In [2]:
DATA_DIR = Path(os.getcwd()).parent.parent / "data"
DATSET = DATA_DIR / "1b_dataset_cleaned.parquet"

In [3]:
df = pd.read_parquet(DATSET)
df["time"] = pd.to_datetime(df["time"])
df["date"] = df["time"].dt.normalize()

In [4]:
# We will be forming our windows and splits based on proper mood readings
# meaning we need at least 1 mood reading to actually form a valid target and the input window

df = df.query("variable == 'mood'")

In [5]:
# unique id + date pairs with calculated mean moods!
unique_subset = ["id", "date"]

df = df.groupby(unique_subset)["value"].mean().reset_index(name="target_mean_mood")

In [6]:
# split by users
unique_ids = df["id"].unique()
permuted_indexes = np.random.choice(
    range(len(unique_ids)), size=len(unique_ids), replace=False
)

train_indexes = permuted_indexes[: int(0.7 * len(permuted_indexes))]
val_indexes = permuted_indexes[
    int(0.7 * len(permuted_indexes)) : int(0.8 * len(permuted_indexes))
]
test_indexes = permuted_indexes[int(0.8 * len(permuted_indexes)) :]

split_map = {}
for idx in train_indexes:
    split_map[unique_ids[idx]] = "train"
for idx in val_indexes:
    split_map[unique_ids[idx]] = "val"
for idx in test_indexes:
    split_map[unique_ids[idx]] = "test"

df["split"] = df["id"].map(split_map)

In [7]:
px.scatter(df, x="date", y="id", color="split", title="Mood prediction targets by day")

In [8]:
mood_target_days = df[["id", "date", "split", "target_mean_mood"]]

In [9]:
mood_target_days.to_parquet(DATA_DIR / "1b_dataset_target_splits.parquet", index=False)